In [18]:
%pip install --upgrade --quiet torch pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

### Data Preparation

In [20]:
data = pd.read_csv('rnn_data.csv')

In [21]:
data.head()

,pitch_type,game_date,batter,pitcher,description,zone,stand,p_throws,balls,strikes,...,is_whiff,is_called_strike,is_ball,in_zone,out_zone,count_state,score_diff_bat,pa_id,prev_pitch_type,seq_len
0,FF,2025-03-18,660271,684007,called_strike,12.0,L,L,0,0,...,False,True,False,False,True,0-0,0,778563_1,START,3
1,SL,2025-03-18,660271,684007,ball,13.0,L,L,0,1,...,False,False,True,False,True,0-1,0,778563_1,FF,3
2,FF,2025-03-18,660271,684007,hit_into_play,5.0,L,L,1,1,...,False,False,False,True,False,1-1,0,778563_1,SL,3
3,FS,2025-03-18,669242,684007,ball,13.0,R,L,0,0,...,False,False,True,False,True,0-0,0,778563_2,START,3
4,FS,2025-03-18,669242,684007,swinging_strike,14.0,R,L,1,0,...,True,False,False,False,True,1-0,0,778563_2,FS,3


In [22]:
data["is_real_pitch"] =  data["pitch_type"].notna() & (data["pitch_type"] != "ABS")

data["target_is_real_pitch"] = data.groupby("pa_id")["is_real_pitch"].shift(-1)

data["y_next_pitch_type"] = data.groupby("pa_id")["pitch_type"].shift(-1)

data_train = data[data["target_is_real_pitch"] == True].copy() 

In [23]:
# randomly select plate appearances to be a part of training and test sets

def split_by_pa_id(df: pd.DataFrame, pa_col="pa_id", ratios=(0.8, 0.2), seed: int=42):
    r_train, r_test = ratios
    assert abs((r_train + r_test) - 1.0) < 1e-9

    pa_ids = df[pa_col].dropna().unique()

    rng = np.random.default_rng(seed)
    rng.shuffle(pa_ids)

    n = len(pa_ids)
    n_train = int(n*r_train)

    train_ids = set(pa_ids[:n_train])
    test_ids = set(pa_ids[n_train:])

    train_df = df[df[pa_col].isin(train_ids)].copy()
    test_df = df[df[pa_col].isin(test_ids)].copy()

    return train_df, test_df, train_ids, test_ids

In [24]:
train_df, test_df, train_ids, test_ids = split_by_pa_id(
    data_train, pa_col="pa_id", ratios=(0.8, 0.2), seed=7
)

/tmp/ipykernel_25098/1463695035.py:10: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(pa_ids)


In [25]:
# Features to be used by the model
FEATURE_SPEC = {
    "target": "y_next_pitch_type",
    "cat_cols": [
        "pitcher", "batter", "stand", "p_throws", "inning_topbot",
        "count_state", "prev_pitch_type"
    ],
    "num_cols": [
        "balls", "strikes", "outs_when_up", "inning", "score_diff_bat",
        "on_1b", "on_2b", "on_3b"
    ],
}

TARGET_COL = FEATURE_SPEC["target"]
CAT_COLS = FEATURE_SPEC["cat_cols"]
NUM_COLS = FEATURE_SPEC["num_cols"]

### RNN Setup

In [26]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

PAD_ID = 0

def build_vocab(values):
    uniq = pd.Series(values.dropna().unique())
    return {v: i for i, v in enumerate(uniq, start=1)}

def encode(series, vocab):
    return series.map(vocab).fillna(PAD_ID).astype(int)

cat_vocabs = {c: build_vocab(train_df[c]) for c in CAT_COLS}
y_vocab    = build_vocab(train_df[TARGET_COL])

def encode_df(df):
    out = df.copy()
    for c in CAT_COLS:
        out[c + "_id"] = encode(out[c], cat_vocabs[c])
    out["y_id"] = encode(out[TARGET_COL], y_vocab)
    for c in NUM_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(np.float32)
    return out

train_enc = encode_df(train_df)
test_enc  = encode_df(test_df)

# make all pitch sequences the same length
def make_fixed_sequences(df, pa_col="pa_id", max_len=8):
    X_cat, X_num, Y = [], [], []

    for _, g in df.groupby(pa_col, sort=False):

        cat = g[[c + "_id" for c in CAT_COLS]].to_numpy(np.int64)     
        num = g[NUM_COLS].to_numpy(np.float32)                        
        y   = g["y_id"].to_numpy(np.int64)                            

        L = min(len(g), max_len)
        cat, num, y = cat[:L], num[:L], y[:L]

        pad = max_len - L
        if pad > 0:
            cat = np.pad(cat, ((0,pad),(0,0)), constant_values=PAD_ID)
            num = np.pad(num, ((0,pad),(0,0)), constant_values=0.0)
            y   = np.pad(y,   (0,pad),         constant_values=PAD_ID)

        X_cat.append(cat); X_num.append(num); Y.append(y)

    return (
        torch.tensor(np.stack(X_cat), dtype=torch.long),
        torch.tensor(np.stack(X_num), dtype=torch.float32),
        torch.tensor(np.stack(Y),     dtype=torch.long),
    )

MAX_LEN = 8
Xc_tr, Xn_tr, Y_tr = make_fixed_sequences(train_enc, max_len=MAX_LEN)
Xc_te, Xn_te, Y_te = make_fixed_sequences(test_enc,  max_len=MAX_LEN)

# Create the Dataset
class PitchSeqDS(Dataset):
    def __init__(self, Xc, Xn, Y):
        self.Xc, self.Xn, self.Y = Xc, Xn, Y
    def __len__(self): return self.Y.size(0)
    def __getitem__(self, i): return self.Xc[i], self.Xn[i], self.Y[i]

train_loader = DataLoader(PitchSeqDS(Xc_tr, Xn_tr, Y_tr), batch_size=64, shuffle=True)
test_loader  = DataLoader(PitchSeqDS(Xc_te, Xn_te, Y_te), batch_size=64, shuffle=False)

# Model Creation
class SimplePitchRNN(nn.Module):
    def __init__(self, cat_vocab_sizes, num_features, emb_dim=16, hidden=128, num_classes=16, pad_id=0):
        super().__init__()
        self.cat_cols = list(cat_vocab_sizes.keys())

        self.embs = nn.ModuleDict({
            col: nn.Embedding(cat_vocab_sizes[col], emb_dim, padding_idx=pad_id)
            for col in self.cat_cols
        })
        
        in_dim = len(self.cat_cols) * emb_dim + num_features
        self.rnn = nn.RNN(in_dim, hidden, batch_first=True)
        self.fc  = nn.Linear(hidden, num_classes)

    def forward(self, x_cat, x_num):
        embs = []
        for j, col in enumerate(self.cat_cols):
            embs.append(self.embs[col](x_cat[:, :, j]))  
        x = torch.cat(embs + [x_num], dim=-1)           
        h, _ = self.rnn(x)                               
        return self.fc(h)                                

cat_vocab_sizes = {c: len(cat_vocabs[c]) + 1 for c in CAT_COLS}  
num_classes = len(y_vocab) + 1                                   

# Model Initialization
model = SimplePitchRNN(cat_vocab_sizes, num_features=len(NUM_COLS), num_classes=num_classes)

In [27]:
from datetime import datetime
from pathlib import Path
import pandas as pd

# Export vocabularies and feature lists with a date-stamped filename.
today = datetime.now().strftime("%Y%m%d")
vocab_dir = Path("vocab")
feature_dir = Path("feature_list")
vocab_dir.mkdir(parents=True, exist_ok=True)
feature_dir.mkdir(parents=True, exist_ok=True)

rows = []
for feature, vocab in cat_vocabs.items():
    for value, idx in vocab.items():
        rows.append({"feature": feature, "value": value, "id": idx, "kind": "categorical"})
for value, idx in y_vocab.items():
    rows.append({"feature": TARGET_COL, "value": value, "id": idx, "kind": "target"})

vocab_path = vocab_dir / f"rnn_vocab_{today}.csv"
pd.DataFrame(rows).to_csv(vocab_path, index=False)

feature_rows = []
for c in CAT_COLS:
    feature_rows.append({"feature": c, "kind": "categorical"})
for c in NUM_COLS:
    feature_rows.append({"feature": c, "kind": "numerical"})
feature_rows.append({"feature": TARGET_COL, "kind": "target"})

feature_path = feature_dir / f"rnn_vocab_{today}.csv"
pd.DataFrame(feature_rows).to_csv(feature_path, index=False)

print(f"Wrote vocab to {vocab_path}")
print(f"Wrote feature list to {feature_path}")


Wrote vocab to vocab/rnn_vocab_20260125.csv
Wrote feature list to feature_list/rnn_vocab_20260125.csv


### Training and Evaluation

In [28]:
import torch
import torch.nn as nn

PAD_ID = 0

@torch.no_grad()
def _count_correct_tokens(logits, y, pad_id=0):

    pred = logits.argmax(dim=-1)  
    mask = (y != pad_id)
    correct = ((pred == y) & mask).sum().item()
    total = mask.sum().item()
    return correct, total

def train_one_epoch(model, loader, optimizer, device, num_classes, pad_id=0):
    model.train()
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

    total_loss = 0.0
    correct = 0
    total = 0

    for x_cat, x_num, y in loader:
        x_cat = x_cat.to(device)
        x_num = x_num.to(device)
        y     = y.to(device)

        logits = model(x_cat, x_num) 

        loss = criterion(
            logits.reshape(-1, num_classes),
            y.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        c, t = _count_correct_tokens(logits.detach(), y, pad_id=pad_id)
        correct += c
        total += t

    avg_loss = total_loss / max(1, len(loader))
    acc = correct / max(1, total)
    return avg_loss, acc

@torch.no_grad()
def evaluate(model, loader, device, num_classes, pad_id=0):
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

    total_loss = 0.0
    correct = 0
    total = 0

    for x_cat, x_num, y in loader:
        x_cat = x_cat.to(device)
        x_num = x_num.to(device)
        y     = y.to(device)

        logits = model(x_cat, x_num)  

        loss = criterion(
            logits.reshape(-1, num_classes),
            y.reshape(-1)
        )
        total_loss += loss.item()

        c, t = _count_correct_tokens(logits, y, pad_id=pad_id)
        correct += c
        total += t

    avg_loss = total_loss / max(1, len(loader))
    acc = correct / max(1, total)
    return avg_loss, acc


In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val_loss = float("inf")

for epoch in range(1, 20):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, device,
        num_classes=num_classes, pad_id=PAD_ID
    )
    test_loss, test_acc = evaluate(
        model, test_loader, device,
        num_classes=num_classes, pad_id=PAD_ID
    )

    improved = test_loss < best_val_loss
    if improved:
        best_val_loss = test_loss
        torch.save(model.state_dict(), "simple_pitch_rnn_best.pt")

    print(
        f"Epoch {epoch:03d} | "
        f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
        f"val loss {test_loss:.4f} acc {test_acc:.3f} "
    )


Epoch 001 | train loss 1.6873 acc 0.368 | val loss 1.5343 acc 0.391 
Epoch 002 | train loss 1.4486 acc 0.410 | val loss 1.4004 acc 0.418 
Epoch 003 | train loss 1.3589 acc 0.426 | val loss 1.3490 acc 0.429 
Epoch 004 | train loss 1.3186 acc 0.434 | val loss 1.3218 acc 0.433 
Epoch 005 | train loss 1.2953 acc 0.439 | val loss 1.3117 acc 0.431 
Epoch 006 | train loss 1.2805 acc 0.444 | val loss 1.3008 acc 0.435 
Epoch 007 | train loss 1.2693 acc 0.448 | val loss 1.2937 acc 0.437 
Epoch 008 | train loss 1.2613 acc 0.450 | val loss 1.2898 acc 0.437 
Epoch 009 | train loss 1.2540 acc 0.452 | val loss 1.2915 acc 0.436 
Epoch 010 | train loss 1.2479 acc 0.455 | val loss 1.2875 acc 0.438 
Epoch 011 | train loss 1.2433 acc 0.456 | val loss 1.2889 acc 0.439 
Epoch 012 | train loss 1.2387 acc 0.458 | val loss 1.2902 acc 0.433 
Epoch 013 | train loss 1.2343 acc 0.459 | val loss 1.2868 acc 0.437 
Epoch 014 | train loss 1.2306 acc 0.462 | val loss 1.2881 acc 0.438 
Epoch 015 | train loss 1.2273 acc 

In [30]:
PAD_ID = 0

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for x_cat, x_num, y in test_loader:
        x_cat = x_cat.to(device)
        x_num = x_num.to(device)
        y     = y.to(device)

        outputs = model(x_cat, x_num)      
        predicted = outputs.argmax(dim=-1) 

        mask = (y != PAD_ID)
        correct += ((predicted == y) & mask).sum().item()
        total += mask.sum().item()

accuracy = 100 * correct / total
print(f"Token Accuracy (no PAD): {accuracy:.2f}%")


Token Accuracy (no PAD): 43.53%


In [31]:
from collections import Counter

model.eval()
counts = Counter()

with torch.no_grad():
    for x_cat, x_num, y in test_loader:
        x_cat = x_cat.to(device)
        x_num = x_num.to(device)

        preds = model(x_cat, x_num).argmax(dim=-1)
        for p in preds.view(-1).tolist():
            if p != 0:
                counts[p] += 1

print(counts.most_common(5))


[(2, 104485), (4, 40353), (8, 33181), (6, 25765), (1, 24201)]


In [32]:
# reverse vocab
id_to_pitch = {v: k for k, v in y_vocab.items()}

for pid, cnt in counts.most_common(5):
    print(id_to_pitch.get(pid, "PAD"), cnt)


FF 104485
SI 40353
FC 33181
CU 25765
SL 24201


In [33]:
model.eval()
correct = 0
total = 0
K = 3

with torch.no_grad():
    for x_cat, x_num, y in test_loader:
        x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
        logits = model(x_cat, x_num)
        topk = logits.topk(K, dim=-1).indices 

        mask = (y != PAD_ID)
        match = (topk == y.unsqueeze(-1)).any(dim=-1)

        correct += (match & mask).sum().item()
        total += mask.sum().item()

print(f"Top-{K} Accuracy: {100*correct/total:.2f}%")


Top-3 Accuracy: 87.52%
